In [ ]:
import ast

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.metrics import f1_score
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests


In [ ]:
INVALID_LABELS = {"mixed", "neutral", "unclear"}

FILES = {
    "MMS-LID-256": "mms_predictions.csv",
    "SpeechBrain": "speechbrain_predictions.csv",
    "Whisper small": "whisper_small_predictions.csv",
    "Whisper medium": "whisper_medium_predictions.csv",
}

PRED_COLS = {
    "MMS-LID-256": "mms_prediction",
    "SpeechBrain": "speechbrain_prediction",
    "Whisper small": "whisper_prediction",
    "Whisper medium": "whisper_med_prediction",
}


def clean_lid_subset(df, split_name):
    true_language = (
        df["langs_present"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    return df[
        df["split"].astype("string").str.strip().str.lower().eq(split_name)
        & df["speech_present"].astype("string").str.strip().str.lower().eq("yes")
        & df["speech_type"].astype("string").str.strip().str.lower().eq("live")
        & true_language.notna()
        & true_language.ne("")
        & ~true_language.str.contains(";", na=False)
        & ~true_language.isin(INVALID_LABELS)
        & ~true_language.eq("cantonese")
    ].copy()


def parse_list(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    try:
        return ast.literal_eval(x)
    except:
        return []


def prepare_model_subset(df, model, pred_col, split_name="dev"):
    clean = clean_lid_subset(df, split_name)

    clean["true_language"] = (
        clean["langs_present"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    clean["pred_language"] = (
        clean[pred_col]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    clean["target_language"] = clean["true_language"]

    if model == "MMS-LID-256":
        clean["pred_language"] = clean["pred_language"].replace({
            "modern greek (1453-)": "greek"
        })

    if model == "SpeechBrain" or model.startswith("Whisper"):
        clean["target_language"] = clean["target_language"].replace({
            "mandarin": "chinese"
        })

    clean["top5_list"] = clean["top5_languages"].apply(parse_list)
    clean["top5_list"] = clean["top5_list"].apply(
        lambda values: [
            str(value).strip().lower()
            for value in values
        ]
    )

    if model == "MMS-LID-256":
        clean["top5_list"] = clean["top5_list"].apply(
            lambda values: [
                "greek" if value == "modern greek (1453-)" else value
                for value in values
            ]
        )

    clean["correct"] = (
        clean["pred_language"] == clean["target_language"]
    )

    return clean


In [ ]:
mms = pd.read_csv(FILES["MMS-LID-256"])
speechbrain = pd.read_csv(FILES["SpeechBrain"])
whisper_small = pd.read_csv(FILES["Whisper small"])
whisper_med = pd.read_csv(FILES["Whisper medium"])

model_subsets = {
    "MMS-LID-256": prepare_model_subset(
        mms,
        "MMS-LID-256",
        PRED_COLS["MMS-LID-256"]
    ),
    "SpeechBrain": prepare_model_subset(
        speechbrain,
        "SpeechBrain",
        PRED_COLS["SpeechBrain"]
    ),
    "Whisper small": prepare_model_subset(
        whisper_small,
        "Whisper small",
        PRED_COLS["Whisper small"]
    ),
    "Whisper medium": prepare_model_subset(
        whisper_med,
        "Whisper medium",
        PRED_COLS["Whisper medium"]
    ),
}

for model, subset in model_subsets.items():
    if subset["file_id"].duplicated().any():
        raise ValueError(f"Duplicate file IDs found for {model}")

common_ids = set.intersection(
    *[
        set(subset["file_id"])
        for subset in model_subsets.values()
    ]
)

for model in model_subsets:
    model_subsets[model] = (
        model_subsets[model][
            model_subsets[model]["file_id"].isin(common_ids)
        ]
        .copy()
        .sort_values("file_id")
        .reset_index(drop=True)
    )

reference_ids = set(model_subsets["MMS-LID-256"]["file_id"])

for model, subset in model_subsets.items():
    assert set(subset["file_id"]) == reference_ids

print("Common development clips:", len(reference_ids))
print("All four models use identical clip IDs.")


In [ ]:
mms_dev = model_subsets["MMS-LID-256"]

# top-1

mms_dev_acc = mms_dev["correct"].mean()

# top-5

mms_dev_top5 = mms_dev.apply(
    lambda r: r["target_language"] in r["top5_list"],
    axis=1
).mean()

# macro-f1

dev_labels = sorted(mms_dev["target_language"].unique())

mms_dev_f1 = f1_score(
    mms_dev["target_language"],
    mms_dev["pred_language"],
    labels=dev_labels,
    average="macro",
    zero_division=0
)

# results

print("MMS-LID-256")
print("Dev N:", len(mms_dev))
print("Dev top-1:", round(mms_dev_acc, 4))
print("Dev top-5:", round(mms_dev_top5, 4))
print("Dev macro-F1:", round(mms_dev_f1, 4))


In [ ]:
sb_dev = model_subsets["SpeechBrain"]

# top-1

sb_dev_acc = sb_dev["correct"].mean()

# top-5

sb_dev_top5 = sb_dev.apply(
    lambda r: r["target_language"] in r["top5_list"],
    axis=1
).mean()

# macro-f1

dev_labels = sorted(sb_dev["target_language"].unique())

sb_dev_f1 = f1_score(
    sb_dev["target_language"],
    sb_dev["pred_language"],
    labels=dev_labels,
    average="macro",
    zero_division=0
)

# results

print("SpeechBrain")
print("Dev N:", len(sb_dev))
print("Dev top-1:", round(sb_dev_acc, 4))
print("Dev top-5:", round(sb_dev_top5, 4))
print("Dev macro-F1:", round(sb_dev_f1, 4))


In [ ]:
ws_dev = model_subsets["Whisper small"]

# top-1

ws_dev_acc = ws_dev["correct"].mean()

# top-5

ws_dev_top5 = ws_dev.apply(
    lambda r: r["target_language"] in r["top5_list"],
    axis=1
).mean()

# macro-f1

dev_labels = sorted(ws_dev["target_language"].unique())

ws_dev_f1 = f1_score(
    ws_dev["target_language"],
    ws_dev["pred_language"],
    labels=dev_labels,
    average="macro",
    zero_division=0
)

# results

print("Whisper small")
print("Dev N:", len(ws_dev))
print("Dev top-1:", round(ws_dev_acc, 4))
print("Dev top-5:", round(ws_dev_top5, 4))
print("Dev macro-F1:", round(ws_dev_f1, 4))


In [ ]:
wm_dev = model_subsets["Whisper medium"]

# top-1

wm_dev_acc = wm_dev["correct"].mean()

# top-5

wm_dev_top5 = wm_dev.apply(
    lambda r: r["target_language"] in r["top5_list"],
    axis=1
).mean()

# macro-f1

dev_labels = sorted(wm_dev["target_language"].unique())

wm_dev_f1 = f1_score(
    wm_dev["target_language"],
    wm_dev["pred_language"],
    labels=dev_labels,
    average="macro",
    zero_division=0
)

# results

print("Whisper medium")
print("Dev N:", len(wm_dev))
print("Dev top-1:", round(wm_dev_acc, 4))
print("Dev top-5:", round(wm_dev_top5, 4))
print("Dev macro-F1:", round(wm_dev_f1, 4))


In [ ]:
comparison_dev = pd.DataFrame({
    "Model": [
        "MMS-LID-256",
        "SpeechBrain",
        "Whisper small",
        "Whisper medium"
    ],
    "N": [
        len(mms_dev),
        len(sb_dev),
        len(ws_dev),
        len(wm_dev)
    ],
    "Dev top-1": [
        mms_dev_acc,
        sb_dev_acc,
        ws_dev_acc,
        wm_dev_acc
    ],
    "Dev top-5": [
        mms_dev_top5,
        sb_dev_top5,
        ws_dev_top5,
        wm_dev_top5
    ],
    "Dev macro-F1": [
        mms_dev_f1,
        sb_dev_f1,
        ws_dev_f1,
        wm_dev_f1
    ]
})

comparison_dev["Dev top-1"] = (
    comparison_dev["Dev top-1"] * 100
).round(2)

comparison_dev["Dev top-5"] = (
    comparison_dev["Dev top-5"] * 100
).round(2)

comparison_dev["Dev macro-F1"] = (
    comparison_dev["Dev macro-F1"].round(4)
)

comparison_dev


In [ ]:
MODEL_ORDER = [
    "MMS-LID-256",
    "SpeechBrain",
    "Whisper small",
    "Whisper medium",
]

model_results = {}

for model in MODEL_ORDER:
    clean = model_subsets[model]

    model_results[model] = (
        clean.groupby("true_language")
        .agg(
            n=("file_id", "size"),
            accuracy=("correct", "mean")
        )
    )

    model_results[model]["accuracy"] *= 100

language_counts = model_results["MMS-LID-256"]["n"]

languages = language_counts[
    language_counts >= 10
].index

heatmap_df = pd.DataFrame(index=languages)

for model in MODEL_ORDER:
    heatmap_df[model] = (
        model_results[model]["accuracy"]
        .reindex(languages)
        .astype(float)
    )

heatmap_df["Average"] = heatmap_df[MODEL_ORDER].mean(axis=1)

heatmap_df = heatmap_df.sort_values(
    "Average",
    ascending=False
)

model_values = heatmap_df[MODEL_ORDER].to_numpy(dtype=float)
average_values = heatmap_df[["Average"]].to_numpy(dtype=float)

language_labels = [
    "Mandarin/Chinese" if lang == "mandarin"
    else lang.title()
    for lang in heatmap_df.index
]

fig = plt.figure(
    figsize=(8.5, 0.55 * len(heatmap_df) + 2.2)
)

gs = fig.add_gridspec(
    1,
    2,
    width_ratios=[4, 1],
    wspace=0.12
)

ax = fig.add_subplot(gs[0])
ax_avg = fig.add_subplot(gs[1], sharey=ax)

im = ax.imshow(
    model_values,
    cmap="viridis",
    aspect="auto",
    vmin=0,
    vmax=100
)

ax_avg.imshow(
    average_values,
    cmap="viridis",
    aspect="auto",
    vmin=0,
    vmax=100
)

for i in range(model_values.shape[0]):
    for j in range(model_values.shape[1]):
        if not np.isnan(model_values[i, j]):
            ax.text(
                j,
                i,
                f"{model_values[i, j]:.1f}",
                ha="center",
                va="center",
                color="black"
            )

    if not np.isnan(average_values[i, 0]):
        ax_avg.text(
            0,
            i,
            f"{average_values[i, 0]:.1f}",
            ha="center",
            va="center",
            color="black"
        )

ax.set_xticks(range(4))
ax.set_xticklabels([
    "MMS-LID",
    "SpeechBrain",
    "Whisper-S",
    "Whisper-M"
])

ax_avg.set_xticks([0])
ax_avg.set_xticklabels(["Average"])

ax.set_yticks(range(len(language_labels)))
ax.set_yticklabels(language_labels)

ax_avg.tick_params(
    axis="y",
    left=False,
    labelleft=False
)

ax.set_xlabel(
    "Model",
    labelpad=20
)

ax.set_ylabel("Language")

cbar = fig.colorbar(
    im,
    ax=[ax, ax_avg],
    fraction=0.035,
    pad=0.04
)

cbar.set_label("Top-1 accuracy (%)")

plt.show()

print(heatmap_df.round(2))


In [ ]:
model_data = {}

for model in MODEL_ORDER:
    model_data[model] = (
        model_subsets[model][
            ["file_id", "true_language", "correct"]
        ]
        .copy()
        .rename(columns={"correct": model})
    )

wm_model = "Whisper medium"

comparators = [
    "MMS-LID-256",
    "SpeechBrain",
    "Whisper small",
]

paired = model_data[wm_model].copy()

for model in comparators:
    paired = paired.merge(
        model_data[model][["file_id", model]],
        on="file_id",
        how="inner"
    )

assert len(paired) == len(reference_ids)

language_counts = paired["true_language"].value_counts()

languages = language_counts[
    language_counts >= 10
].index

test_results = []

for language in languages:

    lang_df = paired[
        paired["true_language"] == language
    ].copy()

    wm = lang_df[wm_model].astype(bool)

    for comparator in comparators:

        other = lang_df[comparator].astype(bool)

        both_correct = (wm & other).sum()
        wm_correct_other_wrong = (wm & ~other).sum()
        wm_wrong_other_correct = (~wm & other).sum()
        both_wrong = (~wm & ~other).sum()

        table = [
            [both_correct, wm_correct_other_wrong],
            [wm_wrong_other_correct, both_wrong]
        ]

        test = mcnemar(
            table,
            exact=True
        )

        test_results.append({
            "language": language,
            "n": len(lang_df),
            "comparator": comparator,
            "whisper_medium_accuracy": wm.mean() * 100,
            "comparator_accuracy": other.mean() * 100,
            "difference_pp": (wm.mean() - other.mean()) * 100,
            "WM_correct_comparator_wrong": wm_correct_other_wrong,
            "WM_wrong_comparator_correct": wm_wrong_other_correct,
            "p_raw": test.pvalue
        })

results = pd.DataFrame(test_results)

results["p_holm"] = multipletests(
    results["p_raw"],
    method="holm"
)[1]

results["significant"] = (
    results["p_holm"] < 0.05
)

results["language"] = results["language"].replace({
    "mandarin": "Mandarin/Chinese"
})

mask = results["language"] != "Mandarin/Chinese"

results.loc[mask, "language"] = (
    results.loc[mask, "language"]
    .str.title()
)

results = results.sort_values(
    ["language", "comparator"]
).reset_index(drop=True)

display_cols = [
    "language",
    "n",
    "comparator",
    "whisper_medium_accuracy",
    "comparator_accuracy",
    "difference_pp",
    "WM_correct_comparator_wrong",
    "WM_wrong_comparator_correct",
    "p_raw",
    "p_holm",
    "significant"
]

print(
    results[display_cols].to_string(
        index=False,
        formatters={
            "whisper_medium_accuracy": "{:.1f}".format,
            "comparator_accuracy": "{:.1f}".format,
            "difference_pp": "{:+.1f}".format,
            "p_raw": "{:.4f}".format,
            "p_holm": "{:.4f}".format,
        }
    )
)

results.to_csv(
    "within_language_mcnemar_results.csv",
    index=False
)
